In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Delta_Project").getOrCreate()
data = [
    (1, "C001", "Laptop", 50000),
    (2, "C002", "Mobile", 15000),
    (3, "C003", "Tablet", 20000),
    (4, "C004", "Laptop", 55000)
]
columns = ["id", "customer_id", "product", "amount"]
df = spark.createDataFrame(data, columns)

In [0]:
df=df.write.format("delta").save("/Volumes/workspace/default/sample_data/delta_tab")

In [0]:
from delta.tables import DeltaTable
deltaTab=DeltaTable.forPath(spark,"/Volumes/workspace/default/sample_data/delta_tab")

In [0]:
data=[(5, "C005", "Camera", 30000)]
columns=["id","customer_id","product","amount"]
new_df=spark.createDataFrame(data,columns)
new_df.write.format("delta").mode("append").save("/Volumes/workspace/default/sample_data/delta_tab")

In [0]:
df=spark.read.format("delta").load("/Volumes/workspace/default/sample_data/delta_tab")

In [0]:
df.display()

id,customer_id,product,amount
1,C001,Laptop,50000
2,C002,Mobile,15000
3,C003,Tablet,20000
4,C004,Laptop,55000
5,C005,Camera,30000


In [0]:
deltaTab=DeltaTable.forPath(spark,"/Volumes/workspace/default/sample_data/delta_tab")

In [0]:
deltaTab.update(
    condition="id=2",
    set={
        "amount":"18000"
    }
)

DataFrame[num_affected_rows: bigint]

In [0]:
deltaTab.delete("id=1")

DataFrame[num_affected_rows: bigint]

In [0]:
df=spark.read.format("delta").load("/Volumes/workspace/default/sample_data/delta_tab")

In [0]:
deltaTab=DeltaTable.forPath(spark,"/Volumes/workspace/default/sample_data/delta_tab")

In [0]:
data = [
    (3, "C003", "Tablet", 22000),  
    (6, "C006", "Watch", 8000)
]
columns=["id","customer_id","product","amount"]
new_data=spark.createDataFrame(data,columns)
new_data.display()



id,customer_id,product,amount
3,C003,Tablet,22000
6,C006,Watch,8000


In [0]:
deltaTab.alias("target").merge(
    new_data.alias("source"),
    "target.id=source.id"
).whenMatchedUpdate(set={
    "amount":"source.amount"
}).whenNotMatchedInsert(values={
    "id":"source.id",
    "customer_id":"source.customer_id",
    "product":"source.product",
    "amount":"source.amount"
}).execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
data = [
    (7, "C007", "Anklets", 17000,"India"),  
]
columns=["id","customer_id","product","amount","country"]
new_data=spark.createDataFrame(data,columns)
new_data.display()



id,customer_id,product,amount,country
7,C007,Anklets,17000,India


In [0]:
new_data.write.format("delta").mode("append").option("mergeSchema","true").save("/Volumes/workspace/default/sample_data/delta_tab")

In [0]:
df=spark.read.format("delta").load("/Volumes/workspace/default/sample_data/delta_tab")

In [0]:
df.display()

id,customer_id,product,amount,country
7,C007,Anklets,17000,India
2,C002,Mobile,18000,null
4,C004,Laptop,55000,null
5,C005,Camera,30000,null
3,C003,Tablet,22000,null
6,C006,Watch,8000,null


In [0]:
old_data=spark.read.format("delta").option("versionAsOf",0).load("/Volumes/workspace/default/sample_data/delta_tab")
old_data.display()

id,customer_id,product,amount
1,C001,Laptop,50000
2,C002,Mobile,15000
3,C003,Tablet,20000
4,C004,Laptop,55000
